# Module 4: AgentCore Evaluations

## Overview

In Module 3, we deployed the Product Catalog Agent to AgentCore with **full observability enabled** — OpenTelemetry traces and GenAI events are already flowing to CloudWatch. In this module, we set up **AgentCore Evaluations** to automatically assess agent performance using both **online** and **on-demand** evaluation methods.

**AgentCore Evaluations** provides automated assessment tools that:
- Use LLM-as-a-Judge techniques to evaluate agent responses
- Support 14 built-in evaluators covering quality, safety, and accuracy
- Integrate seamlessly with CloudWatch GenAI Observability

## Evaluation Types

| Type | Description | Use Case |
|------|-------------|----------|
| **Online Evaluation** | Continuously monitors live production traffic | Production monitoring, quality trends |
| **On-Demand Evaluation** | Targeted assessment of specific traces/spans | Issue investigation, testing custom evaluators |

## Built-in Evaluators

| Level | Evaluators |
|-------|------------|
| **Session** | GoalSuccessRate |
| **Trace** | Coherence, Conciseness, ContextRelevance, Correctness, Faithfulness, Harmfulness, Helpfulness, InstructionFollowing, Refusal, ResponseRelevance, Stereotyping |
| **Tool** | ToolParameterAccuracy, ToolSelectionAccuracy |

## What We'll Do

1. **Setup** — Load deployment info from Module 3 and configure prerequisites
2. **Create Evaluation IAM Role** — Permissions for Bedrock model access, CloudWatch read/write
3. **Online Evaluation** — Create continuous monitoring configuration
4. **Generate Test Data** — Invoke the agent to create evaluation data
5. **On-Demand Evaluation** — Evaluate specific traces programmatically
6. **View Results** — Analyze evaluation results in CloudWatch

## Prerequisites

- Module 3 completed (Product Catalog Agent deployed with observability)
- CloudWatch Transaction Search enabled
- AWS credentials with AgentCore and CloudWatch permissions

**Note:** This notebook retrieves the Product Catalog Agent directly from AgentCore API — you do not need to run Module 3 in the same session.

### Time: ~25 minutes

## Step 1: Setup and Configuration

In [ ]:
# Install dependencies
!pip install -q boto3 bedrock-agentcore

In [10]:
import boto3
import json
import time
import uuid
from datetime import datetime, timezone, timedelta
from botocore.exceptions import ClientError

# Load deployment info from Module 3
try:
    %store -r deployment_info
    %store -r REGION
    print(f"Loaded deployment info from Module 3")
    print(f"  Region: {REGION}")
    print(f"  Runtime ARN: {deployment_info.get('runtime_arn', 'N/A')}")
    print(f"  OTEL Service Name: {deployment_info.get('otel_service_name', 'N/A')}")
    print(f"  User Pool ID: {deployment_info.get('user_pool_id', 'N/A')}")
    print(f"  User Client ID: {deployment_info.get('user_client_id', 'N/A')}")
except:
    print("Could not load deployment info from Module 3")
    print("Using default region — will discover agent from API")
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    deployment_info = None

# Initialize AWS clients
sts_client = boto3.client('sts', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)
logs_client = boto3.client('logs', region_name=REGION)
agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
cognito_client = boto3.client('cognito-idp', region_name=REGION)

ACCOUNT_ID = sts_client.get_caller_identity()['Account']

print(f"\nConfiguration:")
print(f"  Account ID: {ACCOUNT_ID}")
print(f"  Region: {REGION}")

Loaded deployment info from Module 3
  Region: us-west-2
  Runtime ARN: arn:aws:bedrock-agentcore:us-west-2:537124949553:runtime/ecommerce_workshop_product_catalog_agent-2VOibk72lm
  OTEL Service Name: ecommerce-workshop-product-catalog-agent
  User Pool ID: us-west-2_eFyCiUI4k
  User Client ID: 8kgak1kuhbcbn7hpag5b4ucab

Configuration:
  Account ID: 537124949553
  Region: us-west-2


In [11]:
# Retrieve the Product Catalog Agent from Module 3
# This agent was deployed with observability in Module 3

AGENT_RUNTIME_NAME = 'ecommerce_workshop_product_catalog_agent'

def get_product_catalog_agent():
    """
    Find the Product Catalog Agent runtime by name.
    Handles pagination to search through all agents.
    """
    print(f"Searching for Product Catalog Agent: {AGENT_RUNTIME_NAME}\n")
    
    # First try deployment_info from Module 3
    if deployment_info and deployment_info.get('runtime_arn'):
        runtime_id = deployment_info.get('runtime_id')
        try:
            details = agentcore_control_client.get_agent_runtime(agentRuntimeId=runtime_id)
            if details.get('status') in ['READY', 'ACTIVE']:
                print(f"  Found agent from Module 3 deployment info")
                return {
                    'name': details.get('agentRuntimeName', AGENT_RUNTIME_NAME),
                    'id': runtime_id,
                    'arn': deployment_info['runtime_arn'],
                    'status': details.get('status')
                }
        except Exception:
            pass
    
    # Fall back to searching by name
    try:
        next_token = None
        page_count = 0
        
        while True:
            page_count += 1
            params = {}
            if next_token:
                params['nextToken'] = next_token
            
            response = agentcore_control_client.list_agent_runtimes(**params)
            runtimes = response.get('agentRuntimes', [])
            
            print(f"  Page {page_count}: checking {len(runtimes)} agents...")
            
            for runtime in runtimes:
                runtime_name = runtime.get('agentRuntimeName', '')
                if runtime_name == AGENT_RUNTIME_NAME:
                    print(f"  Found Product Catalog Agent!")
                    return {
                        'name': runtime_name,
                        'id': runtime['agentRuntimeId'],
                        'arn': runtime['agentRuntimeArn'],
                        'status': runtime.get('status', 'UNKNOWN')
                    }
            
            next_token = response.get('nextToken')
            if not next_token:
                break
        
        print(f"  Agent not found after {page_count} page(s)")
        return None
    except Exception as e:
        print(f"  Error searching for agent: {e}")
        return None


AGENT_INFO = get_product_catalog_agent()

print("\n" + "=" * 60)
print("PRODUCT CATALOG AGENT FOR EVALUATION")
print("=" * 60)

if AGENT_INFO:
    AGENT_ID = AGENT_INFO['id']
    AGENT_ARN = AGENT_INFO['arn']
    AGENT_NAME = AGENT_INFO['name']
    
    # Derive OTEL service name (same logic as Module 3)
    if deployment_info and deployment_info.get('otel_service_name'):
        OTEL_SERVICE_NAME = deployment_info['otel_service_name']
    else:
        OTEL_SERVICE_NAME = AGENT_NAME.replace('_', '-')
    
    print(f"\n  Name: {AGENT_NAME}")
    print(f"  ID: {AGENT_ID}")
    print(f"  ARN: {AGENT_ARN}")
    print(f"  Status: {AGENT_INFO['status']}")
    print(f"  OTEL Service Name: {OTEL_SERVICE_NAME}")
else:
    print("\nERROR: Product Catalog Agent not found!")
    print("   Please ensure Module 3 was completed successfully.")
    print(f"   Looking for agent named: {AGENT_RUNTIME_NAME}")
    AGENT_ID = None
    AGENT_ARN = None
    OTEL_SERVICE_NAME = None

Searching for Product Catalog Agent: ecommerce_workshop_product_catalog_agent

  Found agent from Module 3 deployment info

PRODUCT CATALOG AGENT FOR EVALUATION

  Name: ecommerce_workshop_product_catalog_agent
  ID: ecommerce_workshop_product_catalog_agent-2VOibk72lm
  ARN: arn:aws:bedrock-agentcore:us-west-2:537124949553:runtime/ecommerce_workshop_product_catalog_agent-2VOibk72lm
  Status: READY
  OTEL Service Name: ecommerce-workshop-product-catalog-agent


In [12]:
# Authenticate with Cognito to get JWT tokens for agent invocation
# The Product Catalog Agent requires bearer_token and access_token in the payload
# to authenticate with the Gateway and determine user role for RBAC

# Load Cognito configuration from Module 3 deployment info
USER_POOL_ID = deployment_info.get('user_pool_id') if deployment_info else None
USER_CLIENT_ID = deployment_info.get('user_client_id') if deployment_info else None
CUSTOMER_EMAIL = 'john.customer@example.com'  # Test user created in Module 3
TEST_PASSWORD = 'Workshop1234!'

# Fallback: discover user_client_id from Cognito if not in deployment_info
if USER_POOL_ID and not USER_CLIENT_ID:
    print("user_client_id not found in deployment_info, discovering from Cognito...")
    try:
        clients_resp = cognito_client.list_user_pool_clients(UserPoolId=USER_POOL_ID, MaxResults=60)
        for client in clients_resp.get('UserPoolClients', []):
            if 'user-client' in client.get('ClientName', ''):
                USER_CLIENT_ID = client['ClientId']
                print(f"  Found user client: {USER_CLIENT_ID} ({client['ClientName']})")
                break
        if not USER_CLIENT_ID:
            # Use the first client without a secret (user-facing clients don't have secrets)
            for client in clients_resp.get('UserPoolClients', []):
                details = cognito_client.describe_user_pool_client(
                    UserPoolId=USER_POOL_ID, ClientId=client['ClientId']
                )
                if not details['UserPoolClient'].get('ClientSecret'):
                    USER_CLIENT_ID = client['ClientId']
                    print(f"  Found user client (no secret): {USER_CLIENT_ID}")
                    break
    except Exception as e:
        print(f"  Error discovering client: {e}")

def get_user_tokens(user_pool_id, client_id, email, password):
    """Get JWT tokens for a Cognito user via ADMIN_USER_PASSWORD_AUTH."""
    try:
        response = cognito_client.admin_initiate_auth(
            UserPoolId=user_pool_id,
            ClientId=client_id,
            AuthFlow='ADMIN_USER_PASSWORD_AUTH',
            AuthParameters={
                'USERNAME': email,
                'PASSWORD': password
            }
        )
        tokens = response.get('AuthenticationResult', {})
        print(f"  Authenticated: {email}")
        return {
            'id_token': tokens.get('IdToken', ''),
            'access_token': tokens.get('AccessToken', ''),
        }
    except Exception as e:
        print(f"  Authentication error: {e}")
        return {}


print("Authenticating test user for agent invocation...\n")

if USER_POOL_ID and USER_CLIENT_ID:
    USER_TOKENS = get_user_tokens(USER_POOL_ID, USER_CLIENT_ID, CUSTOMER_EMAIL, TEST_PASSWORD)
    if USER_TOKENS.get('id_token'):
        print(f"  ID token obtained (length: {len(USER_TOKENS['id_token'])})")
        print(f"  Access token obtained (length: {len(USER_TOKENS['access_token'])})")
    else:
        print("  WARNING: Could not get tokens. Agent invocations may fail.")
        USER_TOKENS = {}
else:
    print("ERROR: Cognito configuration not found in deployment_info.")
    print("  Ensure Module 3 stored user_pool_id and user_client_id.")
    print("  You may need to re-run the Module 3 deployment_info store cell (cell-34).")
    USER_TOKENS = {}

Authenticating test user for agent invocation...

  Authenticated: john.customer@example.com
  ID token obtained (length: 1159)
  Access token obtained (length: 1111)


## Step 2: Create IAM Role for Evaluations

AgentCore Evaluations requires an IAM role with permissions to:
- Invoke Amazon Bedrock models (for LLM-as-Judge evaluation)
- Read traces from Amazon CloudWatch
- Write evaluation results to Amazon CloudWatch

In [13]:
EVALUATION_ROLE_NAME = 'ecommerce-workshop-evaluation-role'

def create_evaluation_role():
    """Create IAM role for AgentCore Evaluations."""
    print(f"Creating evaluation role: {EVALUATION_ROLE_NAME}\n")
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID}
            }
        }]
    }
    
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "BedrockModelAccess",
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream"
                ],
                "Resource": [
                    f"arn:aws:bedrock:{REGION}::foundation-model/*",
                    f"arn:aws:bedrock:*:{ACCOUNT_ID}:inference-profile/*"
                ]
            },
            {
                "Sid": "CloudWatchLogsRead",
                "Effect": "Allow",
                "Action": [
                    "logs:GetLogEvents",
                    "logs:FilterLogEvents",
                    "logs:StartQuery",
                    "logs:GetQueryResults",
                    "logs:DescribeLogGroups",
                    "logs:DescribeLogStreams"
                ],
                "Resource": [
                    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:*"
                ]
            },
            {
                "Sid": "CloudWatchLogsWrite",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents"
                ],
                "Resource": [
                    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/evaluations/*"
                ]
            },
            {
                "Sid": "CloudWatchMetrics",
                "Effect": "Allow",
                "Action": ["cloudwatch:PutMetricData"],
                "Resource": "*",
                "Condition": {
                    "StringEquals": {
                        "cloudwatch:namespace": "Bedrock-AgentCore/Evaluations"
                    }
                }
            },
            {
                "Sid": "XRayRead",
                "Effect": "Allow",
                "Action": [
                    "xray:GetTraceSummaries",
                    "xray:BatchGetTraces",
                    "xray:GetTraceGraph"
                ],
                "Resource": "*"
            }
        ]
    }
    
    try:
        # Check if role already exists
        try:
            response = iam_client.get_role(RoleName=EVALUATION_ROLE_NAME)
            role_arn = response['Role']['Arn']
            print(f"Role already exists: {role_arn}")
            # Update permissions policy
            iam_client.put_role_policy(
                RoleName=EVALUATION_ROLE_NAME,
                PolicyName='evaluation-permissions',
                PolicyDocument=json.dumps(permissions_policy)
            )
            print("Updated permissions policy")
            return role_arn
        except ClientError as e:
            if e.response['Error']['Code'] != 'NoSuchEntity':
                raise
        
        # Create the role
        response = iam_client.create_role(
            RoleName=EVALUATION_ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description='IAM role for AgentCore Evaluations'
        )
        role_arn = response['Role']['Arn']
        print(f"Created role: {role_arn}")
        
        iam_client.put_role_policy(
            RoleName=EVALUATION_ROLE_NAME,
            PolicyName='evaluation-permissions',
            PolicyDocument=json.dumps(permissions_policy)
        )
        print("Attached permissions policy")
        
        print("Waiting for role to propagate...")
        time.sleep(10)
        return role_arn
        
    except Exception as e:
        print(f"Error creating role: {e}")
        return None


EVALUATION_ROLE_ARN = create_evaluation_role()
print(f"\nEvaluation Role ARN: {EVALUATION_ROLE_ARN}")

Creating evaluation role: ecommerce-workshop-evaluation-role

Role already exists: arn:aws:iam::537124949553:role/ecommerce-workshop-evaluation-role
Updated permissions policy

Evaluation Role ARN: arn:aws:iam::537124949553:role/ecommerce-workshop-evaluation-role


## Step 3: Create Online Evaluation Configuration

Online evaluation continuously monitors agent quality using live production traffic. We create an evaluation configuration for the Product Catalog Agent.

**Configuration includes:**
- **Data source**: CloudWatch log groups from our observability-enabled agent
- **Evaluators**: Helpfulness, GoalSuccessRate, ToolSelectionAccuracy, Coherence
- **Sampling**: 100% (for demo purposes; use 10-20% in production)

In [ ]:
ONLINE_EVAL_CONFIG_NAME = 'ecommerce_workshop_product_catalog_eval'
ONLINE_EVAL_CONFIG = None

def get_agent_log_groups(agent_id, endpoint='DEFAULT'):
    """Get CloudWatch log groups for an agent."""
    return [
        f"/aws/bedrock-agentcore/runtimes/{agent_id}-{endpoint}",
        f"/aws/vendedlogs/bedrock-agentcore/{agent_id}"
    ]


if AGENT_ID and OTEL_SERVICE_NAME:
    LOG_GROUPS = get_agent_log_groups(AGENT_ID, 'DEFAULT')
    
    print("Agent Configuration for Evaluation:")
    print(f"  Agent ID: {AGENT_ID}")
    print(f"  OTEL Service Name: {OTEL_SERVICE_NAME}")
    print(f"  Log Groups:")
    for lg in LOG_GROUPS:
        print(f"    - {lg}")
else:
    print("ERROR: Agent not found. Please run Step 1 first.")
    LOG_GROUPS = []


# --- Create online evaluation configuration ---

def create_online_evaluation_config():
    """Create an online evaluation configuration for continuous monitoring."""
    print(f"\nCreating online evaluation config: {ONLINE_EVAL_CONFIG_NAME}\n")
    
    evaluators = [
        {"evaluatorId": "Builtin.Helpfulness"},
        {"evaluatorId": "Builtin.GoalSuccessRate"},
        {"evaluatorId": "Builtin.ToolSelectionAccuracy"},
        {"evaluatorId": "Builtin.Coherence"}
    ]
    
    # Service name format for AgentCore Runtime: <agent-runtime-name>.<endpoint-name>
    service_name = f"{AGENT_NAME}.DEFAULT"
    
    try:
        response = agentcore_control_client.create_online_evaluation_config(
            onlineEvaluationConfigName=ONLINE_EVAL_CONFIG_NAME,
            description="Online evaluation for Product Catalog Agent",
            rule={
                "samplingConfig": {"samplingPercentage": 100.0}  # 100% for demo; use 10-20% in production
            },
            dataSourceConfig={
                "cloudWatchLogs": {
                    "logGroupNames": LOG_GROUPS,
                    "serviceNames": [service_name]
                }
            },
            evaluators=evaluators,
            evaluationExecutionRoleArn=EVALUATION_ROLE_ARN,
            enableOnCreate=True
        )
        
        config_id = response.get('onlineEvaluationConfigId', 'unknown')
        print(f"Created online evaluation config!")
        print(f"  Config ID: {config_id}")
        print(f"  Status: {response.get('status', 'CREATING')}")
        print(f"  Execution Status: {response.get('executionStatus', 'N/A')}")
        
        return {
            'config_name': ONLINE_EVAL_CONFIG_NAME,
            'config_id': config_id,
            'status': response.get('status', 'CREATING'),
            'evaluators': [e['evaluatorId'] for e in evaluators]
        }
    except ClientError as e:
        error_code = e.response['Error']['Code']
        error_msg = e.response['Error']['Message']
        
        if 'ConflictException' in error_code or 'already exists' in error_msg.lower():
            print(f"Online evaluation config already exists, retrieving...")
            try:
                configs = agentcore_control_client.list_online_evaluation_configs()
                for config in configs.get('items', configs.get('onlineEvaluationConfigs', [])):
                    if config.get('onlineEvaluationConfigName') == ONLINE_EVAL_CONFIG_NAME:
                        config_id = config.get('onlineEvaluationConfigId')
                        print(f"  Found existing config: {config_id}")
                        return {
                            'config_name': ONLINE_EVAL_CONFIG_NAME,
                            'config_id': config_id,
                            'status': config.get('status', 'ACTIVE'),
                            'evaluators': [e['evaluatorId'] for e in evaluators]
                        }
            except Exception as list_err:
                print(f"  Error listing configs: {list_err}")
        else:
            print(f"Error creating config: {error_code} - {error_msg}")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None


if AGENT_ARN and EVALUATION_ROLE_ARN and LOG_GROUPS:
    ONLINE_EVAL_CONFIG = create_online_evaluation_config()
    
    if ONLINE_EVAL_CONFIG:
        print(f"\nOnline Evaluation Config:")
        print(f"  Name: {ONLINE_EVAL_CONFIG['config_name']}")
        print(f"  ID: {ONLINE_EVAL_CONFIG['config_id']}")
        print(f"  Status: {ONLINE_EVAL_CONFIG['status']}")
        print(f"  Evaluators: {', '.join(ONLINE_EVAL_CONFIG['evaluators'])}")
    else:
        print("\nWARNING: Could not create online evaluation config.")
        print("On-demand evaluation will still work.")
else:
    print("ERROR: Missing prerequisites. Ensure Steps 1-2 completed successfully.")

## Step 4: Generate Test Data

Invoke the Product Catalog Agent with test queries to generate traces for evaluation. The agent requires JWT tokens for Gateway authentication and RBAC.

In [ ]:
def invoke_agent(agent_arn, prompt, session_id=None):
    """
    Invoke the Product Catalog Agent and return the response.
    Includes JWT tokens required for Gateway authentication and RBAC.
    """
    if not session_id:
        # Generate session ID with min 33 chars (API requirement)
        session_id = f"eval-test-{int(time.time())}-{uuid.uuid4().hex}"
    
    print(f"\nInvoking agent...")
    print(f"  Prompt: {prompt[:80]}..." if len(prompt) > 80 else f"  Prompt: {prompt}")
    print(f"  Session ID: {session_id}")
    
    start_time = time.time()
    
    try:
        # Build payload with JWT tokens (same format as Module 3)
        payload = json.dumps({
            "prompt": prompt,
            "bearer_token": USER_TOKENS.get('id_token', ''),
            "access_token": USER_TOKENS.get('access_token', ''),
            "session_id": session_id
        })
        
        response = agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=agent_arn,
            runtimeSessionId=session_id,
            payload=payload.encode('utf-8'),
            qualifier='DEFAULT'
        )
        
        elapsed = time.time() - start_time
        
        response_body = response.get('response', b'')
        if hasattr(response_body, 'read'):
            response_body = response_body.read()
        if isinstance(response_body, bytes):
            response_body = response_body.decode('utf-8')
        
        try:
            response_data = json.loads(response_body)
        except (json.JSONDecodeError, TypeError):
            response_data = response_body
        
        # Extract response text
        if isinstance(response_data, dict):
            message = response_data.get('response', response_data.get('message', str(response_data)))
        else:
            message = str(response_data)
        
        # Check for agent-level errors
        is_error = (isinstance(response_data, dict) and 
                    response_data.get('status') == 'error')
        
        print(f"  Response received in {elapsed:.2f}s")
        if is_error:
            print(f"  ERROR: {response_data.get('error', 'Unknown error')}")
        else:
            print(f"  Response: {str(message)[:120]}..." if len(str(message)) > 120 else f"  Response: {message}")
        
        return {
            'success': not is_error,
            'session_id': session_id,
            'elapsed_time': elapsed,
            'response': response_data,
            'message': message,
            'timestamp': datetime.now(timezone.utc).isoformat()
        }
    except Exception as e:
        print(f"  Error: {e}")
        return {
            'success': False,
            'session_id': session_id,
            'error': str(e),
            'timestamp': datetime.now(timezone.utc).isoformat()
        }

In [9]:
# Test queries covering different product catalog scenarios
TEST_QUERIES = [
    "Find wireless headphones under $100",
    "What are the details for product PROD-001?",
    "Check inventory availability for PROD-003",
    "Compare products PROD-001 and PROD-002 side by side",
    "Recommend products in the Electronics category under $200",
    "Search for gaming accessories",
]

print("=" * 60)
print("GENERATING TEST DATA FOR EVALUATION")
print("=" * 60)
print(f"\nWill invoke {len(TEST_QUERIES)} queries to generate evaluation data")

INVOCATION_RESULTS = []
SESSION_IDS = []

if AGENT_ARN:
    print(f"\nAgent ARN: {AGENT_ARN}")
    print(f"Agent Status: {AGENT_INFO.get('status', 'UNKNOWN')}")
    
    for i, query in enumerate(TEST_QUERIES, 1):
        print(f"\n{'=' * 60}")
        print(f"Query {i}/{len(TEST_QUERIES)}")
        print(f"{'=' * 60}")
        
        result = invoke_agent(AGENT_ARN, query)
        INVOCATION_RESULTS.append(result)
        time.sleep(2)
    
    # Summary
    print("\n" + "=" * 60)
    print("INVOCATION SUMMARY")
    print("=" * 60)
    successful = sum(1 for r in INVOCATION_RESULTS if r.get('success'))
    print(f"\n{successful}/{len(INVOCATION_RESULTS)} invocations successful")
    
    SESSION_IDS = [r['session_id'] for r in INVOCATION_RESULTS if r.get('success')]
    print(f"Session IDs collected: {len(SESSION_IDS)}")
else:
    print("\nERROR: Agent not available. Please run Step 1 first.")

GENERATING TEST DATA FOR EVALUATION

Will invoke 6 queries to generate evaluation data

Agent ARN: arn:aws:bedrock-agentcore:us-west-2:537124949553:runtime/ecommerce_workshop_product_catalog_agent-2VOibk72lm
Agent Status: READY

Query 1/6

Invoking agent...
  Prompt: Find wireless headphones under $100
  Session ID: eval-test-1770877124-3bf70b03942d424598067b1ee4078cbb
  Response received in 1.02s
  Response: {'status': 'error', 'error': 'the client initialization failed'}

Query 2/6

Invoking agent...
  Prompt: What are the details for product PROD-001?
  Session ID: eval-test-1770877127-105d734c51ac4cb78ec5a6ed229e0a0e
  Response received in 0.67s
  Response: {'status': 'error', 'error': 'the client initialization failed'}

Query 3/6

Invoking agent...
  Prompt: Check inventory availability for PROD-003
  Session ID: eval-test-1770877130-62f295eb9c204315afad8717a2d8fd69
  Response received in 0.63s
  Response: {'status': 'error', 'error': 'the client initialization failed'}


KeyboardInterrupt: 

## Step 5: On-Demand Evaluation

On-demand evaluation allows us to evaluate specific traces programmatically. This is useful for:
- Testing custom evaluators before deploying to online evaluation
- Investigating specific customer-reported issues
- Validating fixes for identified problems

### Process:
1. Download span logs from CloudWatch
2. Call the `Evaluate` API with the spans and evaluator

In [ ]:
def query_logs(log_group_name, query_string, time_range_minutes=60):
    """Query CloudWatch Logs using Logs Insights."""
    start_time = datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)
    end_time = datetime.now(timezone.utc)
    
    try:
        query_response = logs_client.start_query(
            logGroupName=log_group_name,
            startTime=int(start_time.timestamp()),
            endTime=int(end_time.timestamp()),
            queryString=query_string
        )
        query_id = query_response['queryId']
        
        while True:
            result = logs_client.get_query_results(queryId=query_id)
            if result['status'] in ['Complete', 'Failed']:
                break
            time.sleep(1)
        
        if result['status'] == 'Failed':
            return []
        return result.get('results', [])
    except Exception as e:
        print(f"    Query error for {log_group_name}: {e}")
        return []


def is_valid_span(span):
    """Check if a span has all required fields for on-demand evaluation."""
    if not isinstance(span, dict):
        return False
    scope = span.get('scope')
    if not isinstance(scope, dict) or 'name' not in scope:
        return False
    if 'spanId' not in span or 'traceId' not in span:
        return False
    return True


def download_session_spans(agent_id, session_id, time_range_minutes=60):
    """
    Download span logs from CloudWatch for a specific session.
    Returns only valid spans that meet evaluation requirements.
    """
    print(f"\nDownloading spans for session: {session_id}")
    
    all_spans = []
    
    # 1. Query aws/spans log group (primary source for OTEL spans)
    for lg in ["aws/spans", "aws/spans/default"]:
        print(f"  Querying {lg}...")
        try:
            query = "fields @timestamp, @message | sort @timestamp asc | limit 1000"
            results = query_logs(lg, query, time_range_minutes)
            
            session_spans = []
            for row in results:
                for field in row:
                    if field['field'] == '@message':
                        value = field['value'].strip()
                        if value.startswith('{'):
                            try:
                                span = json.loads(value)
                                attrs = span.get('attributes', {})
                                if session_id in str(attrs.get('session.id', '')):
                                    session_spans.append(span)
                            except json.JSONDecodeError:
                                pass
            
            print(f"    Found {len(session_spans)} spans matching session")
            all_spans.extend(session_spans)
            
            if session_spans:
                break
        except Exception as e:
            print(f"    Log group {lg} not accessible: {str(e)[:50]}")
    
    # 2. Query vendedlogs log group
    vendedlogs_group = f"/aws/vendedlogs/bedrock-agentcore/{agent_id}"
    print(f"  Querying {vendedlogs_group}...")
    try:
        query = "fields @timestamp, @message | sort @timestamp asc | limit 1000"
        results = query_logs(vendedlogs_group, query, time_range_minutes)
        
        session_spans = []
        for row in results:
            for field in row:
                if field['field'] == '@message':
                    value = field['value'].strip()
                    if value.startswith('{'):
                        try:
                            span = json.loads(value)
                            attrs = span.get('attributes', {})
                            if session_id in str(attrs.get('session.id', '')):
                                session_spans.append(span)
                        except json.JSONDecodeError:
                            pass
        
        print(f"    Found {len(session_spans)} entries matching session")
        all_spans.extend(session_spans)
    except Exception as e:
        print(f"    Error: {str(e)[:50]}")
    
    # Filter to valid spans only
    valid_spans = [s for s in all_spans if is_valid_span(s)]
    
    print(f"\n  Total entries collected: {len(all_spans)}")
    print(f"  Valid spans (with scope, spanId, traceId): {len(valid_spans)}")
    
    return valid_spans


def run_on_demand_evaluation(session_spans, evaluator_id):
    """
    Run on-demand evaluation using the Evaluate API.
    """
    print(f"\nRunning on-demand evaluation with {evaluator_id}")
    print(f"  Input spans: {len(session_spans)}")
    
    if not session_spans:
        print("  No valid spans to evaluate")
        return {'evaluator_id': evaluator_id, 'error': 'No valid spans', 'results': []}
    
    final_spans = [s for s in session_spans if is_valid_span(s)]
    if not final_spans:
        print("  No valid spans after validation")
        return {'evaluator_id': evaluator_id, 'error': 'No valid spans', 'results': []}
    
    try:
        response = agentcore_client.evaluate(
            evaluatorId=evaluator_id,
            evaluationInput={"sessionSpans": final_spans}
        )
        
        results = response.get('evaluationResults', [])
        print(f"  Evaluation complete: {len(results)} results")
        
        return {
            'evaluator_id': evaluator_id,
            'results': results,
            'timestamp': datetime.now(timezone.utc).isoformat()
        }
    except ClientError as e:
        error_msg = e.response['Error']['Message']
        print(f"  Evaluation error: {error_msg[:200]}")
        return {'evaluator_id': evaluator_id, 'error': error_msg, 'results': []}
    except Exception as e:
        print(f"  Unexpected error: {e}")
        return {'evaluator_id': evaluator_id, 'error': str(e), 'results': []}

In [ ]:
# Wait for spans to be available in CloudWatch
print("Waiting 60 seconds for spans to propagate to CloudWatch...")
print("(Spans typically take 1-2 minutes to appear after invocation)")
time.sleep(60)

In [ ]:
# Run on-demand evaluation on collected sessions
print("=" * 60)
print("ON-DEMAND EVALUATION")
print("=" * 60)

ON_DEMAND_RESULTS = []
session_spans = []

# Evaluators for on-demand evaluation
ON_DEMAND_EVALUATORS = [
    "Builtin.Helpfulness",
    "Builtin.Coherence",
    "Builtin.ToolSelectionAccuracy"
]

if SESSION_IDS and AGENT_ID:
    # Use the first successful session
    test_session_id = SESSION_IDS[0]
    print(f"\nEvaluating session: {test_session_id}")
    
    # Download spans for this session
    session_spans = download_session_spans(AGENT_ID, test_session_id)
    
    if session_spans:
        for evaluator_id in ON_DEMAND_EVALUATORS:
            print(f"\n{'-' * 40}")
            result = run_on_demand_evaluation(session_spans, evaluator_id)
            if result:
                ON_DEMAND_RESULTS.append(result)
            time.sleep(2)
    else:
        print("\nNo spans found for session. Spans may still be propagating.")
        print("Try running this cell again in a few minutes.")
else:
    print("\nNo sessions available for on-demand evaluation.")
    print("Please run Step 4 to generate test data first.")

In [ ]:
# Display on-demand evaluation results
print("=" * 60)
print("ON-DEMAND EVALUATION RESULTS")
print("=" * 60)

if ON_DEMAND_RESULTS:
    for eval_result in ON_DEMAND_RESULTS:
        evaluator_id = eval_result.get('evaluator_id', 'Unknown')
        print(f"\n{evaluator_id}")
        print("-" * 40)
        
        if 'error' in eval_result:
            print(f"   Error: {eval_result['error']}")
            continue
        
        results = eval_result.get('results', [])
        if not results:
            print("   No results returned")
            continue
        
        for i, result in enumerate(results, 1):
            print(f"\n   Result {i}:")
            print(f"   Score: {result.get('value', 'N/A')}")
            print(f"   Label: {result.get('label', 'N/A')}")
            
            explanation = result.get('explanation', '')
            if explanation:
                if len(explanation) > 200:
                    explanation = explanation[:200] + "..."
                print(f"   Explanation: {explanation}")
            
            context = result.get('context', {}).get('spanContext', {})
            if context:
                print(f"   Session: {context.get('sessionId', 'N/A')[:40]}...")
                if 'traceId' in context:
                    print(f"   Trace: {context.get('traceId', 'N/A')[:20]}...")
            
            token_usage = result.get('tokenUsage', {})
            if token_usage:
                print(f"   Tokens: {token_usage.get('inputTokens', 0)} in / {token_usage.get('outputTokens', 0)} out")
else:
    print("\nNo on-demand evaluation results available.")
    print("Possible reasons:")
    print("  - No spans were found in CloudWatch")
    print("  - Spans are still propagating (wait a few minutes and retry)")

## Step 6: Evaluate a Specific Trace

This demonstrates how to target a specific trace ID for evaluation. This is useful for investigating individual interactions that were flagged by online evaluation or reported by users.

**Note:** AgentCore traces are delivered to CloudWatch GenAI Observability via log delivery (to the `aws/spans` log group), not traditional X-Ray trace storage.

In [ ]:
# Evaluate a specific trace using trace ID from the spans
print("=" * 60)
print("TRACE-SPECIFIC EVALUATION")
print("=" * 60)

TRACE_EVAL_RESULT = None

if session_spans:
    # Extract unique trace IDs
    trace_ids_in_spans = set()
    for span in session_spans:
        trace_id = span.get('traceId')
        if trace_id:
            trace_ids_in_spans.add(trace_id)
    
    print(f"\nFound {len(trace_ids_in_spans)} unique trace ID(s) in session spans:")
    for tid in list(trace_ids_in_spans)[:5]:
        print(f"  - {tid}")
    
    if trace_ids_in_spans:
        target_trace_id = list(trace_ids_in_spans)[0]
        print(f"\nEvaluating trace: {target_trace_id}")
        
        try:
            response = agentcore_client.evaluate(
                evaluatorId="Builtin.Helpfulness",
                evaluationInput={"sessionSpans": session_spans},
                evaluationTarget={"traceIds": [target_trace_id]}
            )
            
            results = response.get('evaluationResults', [])
            
            if results:
                result = results[0]
                print(f"\nTrace Evaluation Complete")
                print(f"   Evaluator: Builtin.Helpfulness")
                print(f"   Score: {result.get('value', 'N/A')}")
                print(f"   Label: {result.get('label', 'N/A')}")
                
                explanation = result.get('explanation', '')
                if explanation:
                    print(f"   Explanation: {explanation[:300]}..." if len(explanation) > 300 else f"   Explanation: {explanation}")
                
                TRACE_EVAL_RESULT = result
            else:
                print("\nNo evaluation results returned for this trace")
        except Exception as e:
            print(f"\nTrace evaluation error: {e}")
    else:
        print("\nNo trace IDs found in session spans")
else:
    print("\nNo session spans available.")
    print("Please run Steps 4-5 first to download spans.")

## Step 7: View Evaluation Results in CloudWatch

Evaluation results are stored in CloudWatch and can be viewed in:
1. **GenAI Observability Dashboard** — Aggregated scores and trends
2. **CloudWatch Metrics** — Evaluation score metrics
3. **CloudWatch Logs** — Detailed evaluation results

In [ ]:
def check_evaluation_log_group():
    """Check if evaluation results log group exists."""
    if not ONLINE_EVAL_CONFIG:
        return None
    config_id = ONLINE_EVAL_CONFIG.get('config_id')
    log_group = f"/aws/bedrock-agentcore/evaluations/results/{config_id}"
    try:
        logs_client.describe_log_groups(logGroupNamePrefix=log_group)
        return log_group
    except:
        return None


# Check evaluation results
print("=" * 60)
print("CLOUDWATCH EVALUATION RESULTS")
print("=" * 60)

eval_log_group = check_evaluation_log_group()

if eval_log_group:
    print(f"\nEvaluation log group found: {eval_log_group}")
    
    # Query for results
    try:
        query = "fields @timestamp, @message | filter @message like /evaluation/ | sort @timestamp desc | limit 20"
        start_time = datetime.now(timezone.utc) - timedelta(hours=1)
        end_time = datetime.now(timezone.utc)
        
        query_response = logs_client.start_query(
            logGroupName=eval_log_group,
            startTime=int(start_time.timestamp()),
            endTime=int(end_time.timestamp()),
            queryString=query
        )
        
        while True:
            result = logs_client.get_query_results(queryId=query_response['queryId'])
            if result['status'] in ['Complete', 'Failed']:
                break
            time.sleep(1)
        
        eval_logs = result.get('results', [])
        if eval_logs:
            print(f"Found {len(eval_logs)} evaluation result entries")
        else:
            print("No evaluation results yet. Online evaluation results will appear")
            print("after the evaluation job processes agent traces.")
    except Exception as e:
        print(f"Error querying logs: {e}")
else:
    print("\nEvaluation log group not found yet.")
    print("Online evaluation results will be created after the first evaluation runs.")

In [ ]:
# CloudWatch console links
print("\n" + "=" * 70)
print("CLOUDWATCH CONSOLE LINKS")
print("=" * 70)

print(f"\nGenAI Observability Dashboard (with Evaluations):")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore")

print(f"\nCloudWatch Metrics (Evaluation Scores):")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#metricsV2:graph=~();namespace=Bedrock-AgentCore/Evaluations")

if ONLINE_EVAL_CONFIG:
    config_id = ONLINE_EVAL_CONFIG.get('config_id', '')
    print(f"\nEvaluation Results Log Group:")
    print(f"  /aws/bedrock-agentcore/evaluations/results/{config_id}")

print(f"\nX-Ray Traces:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:traces")

## Step 8: Summary

In [ ]:
print("\n" + "=" * 70)
print("MODULE 4 COMPLETE: AGENTCORE EVALUATIONS")
print("=" * 70)

print("\n1. SETUP")
print(f"   Evaluation IAM role: {EVALUATION_ROLE_NAME}")
print(f"   Product Catalog Agent: {AGENT_NAME if AGENT_INFO else 'NOT FOUND'}")

print("\n2. ONLINE EVALUATION")
if ONLINE_EVAL_CONFIG:
    print(f"   Config: {ONLINE_EVAL_CONFIG.get('config_name')}")
    print(f"   Config ID: {ONLINE_EVAL_CONFIG.get('config_id')}")
    print(f"   Status: {ONLINE_EVAL_CONFIG.get('status')}")
    print(f"   Evaluators: Helpfulness, GoalSuccessRate, ToolSelectionAccuracy, Coherence")
else:
    print("   Not configured")

print("\n3. TEST INVOCATIONS")
successful_invocations = sum(1 for r in INVOCATION_RESULTS if r.get('success'))
print(f"   Invocations: {successful_invocations}/{len(INVOCATION_RESULTS)} successful")
print(f"   Sessions collected: {len(SESSION_IDS)}")

print("\n4. ON-DEMAND EVALUATION")
if ON_DEMAND_RESULTS:
    print(f"   Evaluations run: {len(ON_DEMAND_RESULTS)}")
    for result in ON_DEMAND_RESULTS:
        if 'error' not in result:
            num_results = len(result.get('results', []))
            print(f"      - {result['evaluator_id']}: {num_results} results")
else:
    print("   No on-demand evaluations completed")

print("\n" + "=" * 70)
print("KEY TAKEAWAYS")
print("=" * 70)
print("""
1. Online Evaluation:
   - Continuously monitors agent quality with live traffic
   - Results automatically stored in CloudWatch
   - Scores published as CloudWatch metrics for alarming

2. On-Demand Evaluation:
   - Download spans from CloudWatch log groups
   - Call Evaluate API with specific evaluator
   - Can target specific traces or spans for investigation

3. Built-in Evaluators:
   - Session-level: GoalSuccessRate
   - Trace-level: Helpfulness, Coherence, Correctness, etc.
   - Tool-level: ToolSelectionAccuracy, ToolParameterAccuracy

4. Integration with Observability (Module 3):
   - View evaluations in GenAI Observability dashboard
   - Correlate evaluation scores with traces and logs
   - Set CloudWatch alarms on evaluation metrics
""")
print("=" * 70)

In [ ]:
# Store variables for potential use in other modules
%store ONLINE_EVAL_CONFIG
%store EVALUATION_ROLE_ARN
%store SESSION_IDS
%store ON_DEMAND_RESULTS

print("Variables stored for use in subsequent modules")

---

## Cleanup (Optional)

Run the cell below to delete evaluation resources. **Only run this when you're done with the workshop.**

In [ ]:
# UNCOMMENT AND RUN TO CLEAN UP EVALUATION RESOURCES

# def cleanup_evaluation_resources(delete_config=False, delete_role=False):
#     """Clean up evaluation resources."""
#     print("\nCLEANUP - This will delete resources!")
#
#     if delete_config and ONLINE_EVAL_CONFIG:
#         config_id = ONLINE_EVAL_CONFIG.get('config_id')
#         print(f"\nDeleting online evaluation config: {config_id}")
#         try:
#             agentcore_control_client.delete_online_evaluation_config(
#                 onlineEvaluationConfigId=config_id
#             )
#             print("Config deleted")
#         except Exception as e:
#             print(f"Error: {e}")
#
#     if delete_role:
#         print(f"\nDeleting IAM role: {EVALUATION_ROLE_NAME}")
#         try:
#             policies = iam_client.list_role_policies(RoleName=EVALUATION_ROLE_NAME)
#             for policy_name in policies.get('PolicyNames', []):
#                 iam_client.delete_role_policy(
#                     RoleName=EVALUATION_ROLE_NAME,
#                     PolicyName=policy_name
#                 )
#             iam_client.delete_role(RoleName=EVALUATION_ROLE_NAME)
#             print("Role deleted")
#         except Exception as e:
#             print(f"Error: {e}")
#
# cleanup_evaluation_resources(delete_config=True, delete_role=True)

---

## Module 4 Summary

You set up **AgentCore Evaluations** to automatically assess the Product Catalog Agent's performance.

### What Was Created

| Component | Description |
|-----------|-------------|
| **Evaluation IAM Role** | Permissions for Bedrock model access and CloudWatch read/write |
| **Online Evaluation Config** | Continuous monitoring with Helpfulness, GoalSuccessRate, ToolSelectionAccuracy, Coherence |
| **On-Demand Evaluations** | Targeted assessment of specific sessions and traces |

### Evaluation Types

| Type | How It Works | When to Use |
|------|-------------|-------------|
| **Online** | Samples live traffic, runs LLM-as-Judge continuously | Production quality monitoring |
| **On-Demand** | Downloads spans, calls Evaluate API programmatically | Issue investigation, testing evaluators |

### Integration with Module 3 Observability

| Module 3 Component | Module 4 Usage |
|-------------------|----------------|
| OTEL traces (X-Ray) | Data source for online evaluation |
| GenAI events (vendedlogs) | Span data for on-demand evaluation |
| OTEL service name | Filtering traces to this specific agent |
| CloudWatch Log Delivery | Routes telemetry data to evaluation pipeline |

### Workshop Complete!

You have now completed the full agent development lifecycle:
1. **Module 01**: Built a local prototype with golden dataset
2. **Module 02**: Established offline evaluation baseline
3. **Module 03**: Deployed to production with RBAC and observability
4. **Module 04**: Set up online and on-demand evaluation for continuous quality monitoring